# Exploración de Datos (EDA) — Lending Club Loan Data

**Objetivo:** Comprender la estructura del dataset, distribución de la variable objetivo, calidad de los datos y relaciones entre variables antes del modelado.

---

## 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import math
import warnings
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')

print('✅ Librerías cargadas')

## 2. Carga del Dataset

> Ajusta `CSV_PATH` según tu entorno. Usa `nrows=None` para cargar el dataset completo.

In [ ]:
CSV_PATH = 'accepted_2007_to_2018Q4.csv'

df_raw = pd.read_csv(CSV_PATH, nrows=300_000, low_memory=False)

print(f'Filas   : {df_raw.shape[0]:,}')
print(f'Columnas: {df_raw.shape[1]:,}')
df_raw.head(3)

## 3. Variable Objetivo: `default`

Se filtran únicamente préstamos **Fully Paid** o **Charged Off** y se crea la variable binaria `default`.

In [ ]:
df = df_raw[df_raw['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['default'] = df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)
df = df.drop(columns='loan_status')

counts = df['default'].value_counts()
pcts   = df['default'].value_counts(normalize=True) * 100

print(f'Registros tras filtrado: {len(df):,}')
print(f'  Fully Paid  (0): {counts[0]:,}  ({pcts[0]:.1f}%)')
print(f'  Charged Off (1): {counts[1]:,}  ({pcts[1]:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(['Fully Paid (0)', 'Charged Off (1)'], counts.values,
                   color=['#27ae60', '#e74c3c'], edgecolor='white', width=0.5)
for bar, pct in zip(bars, pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Distribución de la Variable Objetivo', fontweight='bold')
axes[0].set_ylabel('Número de préstamos')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[1].pie(counts.values, labels=['Fully Paid (0)', 'Charged Off (1)'],
            autopct='%1.1f%%', colors=['#27ae60', '#e74c3c'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Proporción de Clases', fontweight='bold')

plt.suptitle('Variable objetivo — default', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_01_target.png', bbox_inches='tight')
plt.show()

print('⚠️  Dataset desbalanceado — considerar class_weight o SMOTE en el modelado.')

La variable `default` presenta un **desequilibrio significativo**:
- **Clase mayoritaria (0 — Fully Paid):** ~80% de los registros.  
- **Clase minoritaria (1 — Charged Off):** ~20% de los registros.  

Este desbalance debe tenerse en cuenta al entrenar el modelo.

## 4. Tipos de Datos y Valores Faltantes

### 4.1 Tipos de datos

In [ ]:
type_summary = df.dtypes.value_counts().rename_axis('Tipo').reset_index(name='Cantidad')
print("Distribución de tipos de datos:")
print(type_summary.to_string(index=False))

### 4.2 Diagnóstico de valores faltantes

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

print(f'Columnas con valores faltantes: {len(missing_pct)} de {df.shape[1]}')
missing_pct.head(20)

In [ ]:
plt.figure(figsize=(15, 7))
sns.barplot(x=missing_pct.index, y=missing_pct.values,
            hue=missing_pct.index, palette='Reds_r', legend=False)
plt.axhline(50, color='#c0392b', linestyle='--', linewidth=1.5, label='Umbral 50%')
plt.title('Porcentaje de Valores Faltantes por Columna', fontsize=14, fontweight='bold')
plt.xlabel('Columna')
plt.ylabel('% de valores faltantes')
plt.xticks(rotation=90, fontsize=7)
plt.legend()
plt.tight_layout()
plt.savefig('eda_02_missing_before.png', bbox_inches='tight')
plt.show()

### 4.3 Eliminar columnas con más del 50% de nulos

Columnas con más del 50% de valores faltantes no aportan información útil y no pueden imputarse sin introducir sesgo significativo.

In [ ]:
threshold = 0.50
missing_ratio = df.isnull().sum() / len(df)
cols_to_drop = missing_ratio[missing_ratio > threshold].index

df = df.drop(columns=cols_to_drop)

print(f'Columnas eliminadas (>{int(threshold*100)}% nulos): {len(cols_to_drop)}')
print(f'Columnas restantes: {df.shape[1]}')

Verificamos los nulos restantes tras la eliminación:

In [ ]:
missing_pct2 = (df.isnull().sum() / len(df) * 100)
missing_pct2 = missing_pct2[missing_pct2 > 0].sort_values(ascending=False)

print(f'Columnas con nulos restantes: {len(missing_pct2)}')

plt.figure(figsize=(15, 6))
sns.barplot(x=missing_pct2.index, y=missing_pct2.values,
            hue=missing_pct2.index, palette='Oranges_r', legend=False)
plt.title('Valores Faltantes Restantes (tras eliminar >50%)', fontsize=14, fontweight='bold')
plt.xlabel('Columna')
plt.ylabel('% de valores faltantes')
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.savefig('eda_03_missing_after.png', bbox_inches='tight')
plt.show()

### 4.4 Imputación de variables categóricas

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns
missing_cat = df[cat_cols].isnull().sum()
missing_cat = missing_cat[missing_cat > 0]

print('Datos faltantes en variables categóricas:')
print(missing_cat.to_string())

In [ ]:
df_imp = df.copy()

# Imputar con "Unknown" — campos de texto libre sin valor predictivo directo
cols_unknown = [c for c in ['emp_title', 'title', 'zip_code',
                             'last_pymnt_d', 'last_credit_pull_d'] if c in df_imp.columns]
for col in cols_unknown:
    df_imp[col] = df_imp[col].fillna('Unknown')

print(f'✅ Imputadas con "Unknown": {cols_unknown}')

In [ ]:
# emp_length requiere tratamiento especial: convertir texto a numérico
if 'emp_length' in df_imp.columns:
    df_imp['emp_length'] = (df_imp['emp_length']
                            .replace('10+ years', '10')
                            .replace('< 1 year', '0')
                            .str.replace(' years', '', regex=False)
                            .str.replace(' year', '', regex=False))
    df_imp['emp_length'] = pd.to_numeric(df_imp['emp_length'], errors='coerce')
    mediana_emp = df_imp['emp_length'].median()
    df_imp['emp_length'] = df_imp['emp_length'].fillna(mediana_emp)
    print(f'✅ emp_length convertida a numérico — mediana imputada: {mediana_emp}')
    print(df_imp['emp_length'].value_counts().sort_index().to_string())

### 4.5 Imputación de variables numéricas

In [ ]:
num_cols = df_imp.select_dtypes(include=['number']).columns
missing_num = df_imp[num_cols].isnull().sum()
missing_num = missing_num[missing_num > 0]

print('Datos faltantes en variables numéricas:')
print(missing_num.to_string())

In [ ]:
# Imputar con la mediana — robusta ante outliers
imputer = SimpleImputer(strategy='median')
df_imp[num_cols] = imputer.fit_transform(df_imp[num_cols])

total_nulos = df_imp.isnull().sum().sum()
print(f'✅ Imputación con mediana completada')
print(f'   Nulos restantes en el dataset: {total_nulos}')

## 5. Selección de Variables para el Análisis Visual

Se trabaja con las variables numéricas más relevantes desde el punto de vista financiero y crediticio.

In [ ]:
NUM_COLS = [c for c in ['loan_amnt', 'int_rate', 'fico_range_low', 'fico_range_high',
                         'annual_inc', 'dti', 'installment', 'open_acc',
                         'pub_rec', 'revol_util', 'mort_acc', 'avg_cur_bal']
            if c in df_imp.columns]

print(f'Variables seleccionadas ({len(NUM_COLS)}): {NUM_COLS}')
df_imp[NUM_COLS + ['default']].describe().round(2)

## 6. Histogramas de Variables Numéricas

Se comparan las distribuciones entre **Fully Paid** (0) y **Charged Off** (1). Las diferencias visibles entre clases son indicadores del poder predictivo de cada variable.

In [ ]:
n_cols_g = 3
n_rows_g = math.ceil(len(NUM_COLS) / n_cols_g)

fig, axes = plt.subplots(n_rows_g, n_cols_g, figsize=(16, n_rows_g * 4))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    for label, color, name in [(0,'#27ae60','Fully Paid'), (1,'#e74c3c','Charged Off')]:
        subset = df_imp[df_imp['default'] == label][col].dropna()
        axes[i].hist(subset, bins=40, alpha=0.55, color=color, label=name, density=True)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Densidad')
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Histogramas de Variables Numéricas por Clase', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_04_histograms.png', bbox_inches='tight')
plt.show()

## 7. Boxplots por Clase

Permiten identificar diferencias en la tendencia central y la dispersión entre clases. Los outliers extremos se recortan (percentiles 1–99) para mejorar la legibilidad.

In [ ]:
fig, axes = plt.subplots(n_rows_g, n_cols_g, figsize=(16, n_rows_g * 4))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    data_plot = df_imp[['default', col]].dropna()
    q_lo, q_hi = data_plot[col].quantile([0.01, 0.99])
    data_plot = data_plot[(data_plot[col] >= q_lo) & (data_plot[col] <= q_hi)]

    sns.boxplot(data=data_plot, x=data_plot['default'].astype(str), y=col,
                palette={'0': '#27ae60', '1': '#e74c3c'}, ax=axes[i],
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xticklabels(['Fully Paid', 'Charged Off'])
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Boxplots por Clase (sin outliers extremos)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_05_boxplots.png', bbox_inches='tight')
plt.show()

## 8. Correlaciones

### 8.1 Correlación con la variable objetivo

In [ ]:
corr_target = (df_imp[NUM_COLS + ['default']]
               .corr()['default']
               .drop('default')
               .sort_values(key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr_target.values]
ax.barh(corr_target.index, corr_target.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlación de Pearson con default')
ax.set_title('Correlación de cada variable con default', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('eda_06_corr_target.png', bbox_inches='tight')
plt.show()

print(corr_target.to_string())

### 8.2 Matriz de correlación completa

In [ ]:
corr_matrix = df_imp[NUM_COLS + ['default']].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            ax=ax, annot_kws={'size': 8})
ax.set_title('Matriz de Correlación — Variables Numéricas + default',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_07_heatmap.png', bbox_inches='tight')
plt.show()

## 9. Conclusiones del EDA

In [ ]:
total    = len(df_imp)
n_def    = int(df_imp['default'].sum())
pct_def  = n_def / total * 100
top_corr = corr_target.index[0]

print('=' * 55)
print('       RESUMEN EDA — LENDING CLUB')
print('=' * 55)
print(f'  Registros analizados  : {total:>10,}')
print(f'  Fully Paid  (0)       : {total-n_def:>10,}  ({100-pct_def:.1f}%)')
print(f'  Charged Off (1)       : {n_def:>10,}  ({pct_def:.1f}%)')
print()
print(f'  Variable más correlacionada con default: {top_corr}')
print()
print('  Tratamiento de nulos:')
print(f'   - Columnas eliminadas (>50% nulos): calculado en sección 4.3')
print( '   - Categóricas: imputadas con "Unknown" o mediana (emp_length)')
print( '   - Numéricas: imputadas con la mediana (robusto ante outliers)')
print()
print('  Hallazgos principales:')
print(f'   - Desbalance de clases: {pct_def:.1f}% de defaults')
print( '   - int_rate y fico son los predictores más fuertes')
print( '   - Outliers presentes en annual_inc y loan_amnt')
print('=' * 55)